<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_week02_1_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 수상작 리뷰

# 수상작 소개

**축구선수의 유망 여부 예측 AI 해커톤**  

(https://dacon.io/competitions/official/236031/codeshare/7094?page=1&dtype=recent)

목표: 16세부터 21세의 축구 선수들의 유망 여부를 예측하는 AI 알고리즘 개발

설명: 축구 선수 데이터를 이용해 선수가 유망한 선수인지를 판단하는 스카우터 역할의 AI 모델(16세부터 21세의 축구 선수의 정보로부터 유망 여부를 이진 분류)제작하기.

평가산식: Macro f1 score



---


## 코드 흐름 요약

- Automl 모듈 중 하나인 LightAutoml을 사용
- 파라미터 설정은 직접 값을 바꿔가며 제출

### 코드 흐름

1. 라이브러리 임포트
- 데이터 핸들링 툴
- 데이터 시각화 툴
- 모듈 임포트
- 하이퍼파라미터 설정
- seed 설정

2. 데이터 로드
- head 확인
*EDA 과정 언급 X ...

3. 데이터 전처리
- train, test split
  - 그후에 size, head 확인

4. 모델링
- metric & task setting (!!)
: 점수 매기는 방식, 문제 유형 정의
- 하이퍼 파라미터 세팅 및 학습(LGB, CB(CatBoost))
- 앙상블
- 교차검증(CV): OOF
- 피처 중요도 확인: **Fast/Accurate 방식** 각각 그래프 그려서 확인.
- **OOF Score/HOLDOUT Score** 점수 산출 후 비교
- 모든 train data로 학습 - 학습, 예측, 구조 확인, OOF 점수 확인, 저장 후 제출.

## 새롭게 알게 된 내용 / 어려운 내용 / 배울 점 등 정리 및 주요 코드 3줄 이상 작성

- 피처 중요도 확인 시 **Fast/Accurate 방식** 각각 그래프 그려서 확인
  - fast 방식: 모델이 학습하는 과정에서 각 변수가 얼마나 자주 사용되었는지를 계산. 속도가 매우 빠르지만, 때로는 덜 중요한 변수가 높게 평가될 수 있는 왜곡이 생기기도 함.
  - accurate 방식: 실제 데이터를 하나씩 빼보거나 변형해 보면서 "이 변수가 없으면 점수가 얼마나 떨어지는지" 직접 확인. 시간은 오래 걸리지만 훨씬 정확함.

- **OOF Score/HOLDOUT Score** 점수 산출 후 비교
  - OOF (Out-Of-Fold) Score
    - 의미: 학습 데이터 내에서 데이터를 쪼개어(K-Fold), 모델이 학습할 때 보지 않았던 데이터로 계산한 점수
    - 목적: "우리 모델이 가진 데이터 안에서는 얼마나 일반적인 성능을 내는가?"를 확인
  - HOLDOUT Score
    - 의미: 학습 과정에 **아예 참여하지 않은 별도의 테스트용 데이터(te_data)**로 낸 점수
    - 목적: "완전히 새로운 데이터(실제 현장 데이터)가 들어왔을 때도 잘 작동하는가?"를 최종 점검
  - OOF 점수와 HOLDOUT 점수 비교.
  - 예) OOF 점수는 좋은데 HOLDOUT 점수가 너무 나쁘다면? 과적합(Overfitting)된 것이므로 모델을 수정해야 함.

  ---

- seed 고정... 변수를 따로 지정하고 `np.random.seed()`로 작성
- `torch.set_num_threads(N_THREADS)`: 파이토치(PyTorch)가 연산을 할 때 사용할 CPU 코어(Thread)의 개수를 제한   
  - 여러 모델을 동시에 돌리거나, 서버 자원 효율적으로 나눠 써야할 때 사용.  
  - CPU 자원을 전부 다 끌어다 쓰면 시스템이 버벅거릴 수 있어서, N_THREADS에 지정된 개수(예: 4개, 8개)만큼만 사용하겠다는 뜻




In [1]:
# N_THREADS = 20
# RANDOM_STATE = 42

# np.random.seed(RANDOM_STATE)
# torch.set_num_threads(N_THREADS)

자동화된 라이브러리(AutoML) 특유 설정  

**Task Setting: 어떤 문제인가**  
`task = Task('reg', )`
- Task : 작업 종류
- 'reg' = regression (회귀: 연속적 숫자 맞추기)
- 'binary', 'multiclass' 등도 있음

** Metric Setting: 어떻게 점수를 매길 것인가? **  

** Roles: 주인공 결정, 컬럼들의 역할 정하기 **  
- `'target': TARGET_NAME'` : 예측해야 할 칼럼 지정(Y)
- `'drop': ['ID']` : 아이디 칼럼 제외



In [2]:
### metric & task setting 단계 ###

'''
def rmse(y_true, y_pred, **kwargs):
    return mean_squared_error(y_true, y_pred, squared = False, **kwargs)

task = Task('reg', )

roles = {'target': TARGET_NAME,
         'drop': ['ID']
         }
'''

" \ndef rmse(y_true, y_pred, **kwargs):\n    return mean_squared_error(y_true, y_pred, squared = False, **kwargs)\n\ntask = Task('reg', )\n\nroles = {'target': TARGET_NAME,\n         'drop': ['ID']\n         } \n"

의문&답 도출
- 왜 이진분류 목적에서 회귀를 사용했는가?
  - 연속적 수치를 확인함으로써 더 정밀하게 확률 계산을 하기 위해 회귀로 접근하기도 함.
  - 나중에 특정 임계값 기준으로 나누면, 처음부터 분류로 모델링한 것보다 더 유연하게 F1-score를 최적화할 수 있는 경우가 多
  - 축구 도메인 특성상: 순위가 중요할 수 있음. 회귀 모델에서는 미세한 실력차를 숫자로 보존하므로, 여러 모델 앙상블 할 때 결합력 더 좋을 수 있음

- 왜 Macro F1 대신 RMSE를 지정했지?
  - Macro F1은 클래스별 균형이 중요한 지표. 단순히 0,1 이진분류로 학습시 데이터 많은 쪽으로 편향되기 쉬움.
  - RMSE로 학습하며 정답과의 거리 좁히는 방식으로 훈련하면 미세한 패턴 더 잘 학습할 수 있음.
  - 임계값 조절만 잘 해주면 F1-score 더 높일 수 있음

  ---

  


In [3]:
'''
%%time

automl = TabularAutoML(task = task,
                       timeout = TIMEOUT,
                       cpu_limit = N_THREADS,
                       reader_params = {'n_jobs': N_THREADS, 'cv': N_FOLDS, 'random_state': RANDOM_STATE},
                       general_params = {'use_algos': [['lgb', 'cb']]},
                       lgb_params = {'default_params': lgb_params, 'freeze_defaults': True},
                       cb_params = {'default_params': cb_params, 'freeze_defaults': True}
                      ) # 모델 생성

RD = ReportDeco(output_path = 'tabularAutoML_model_report') # 학습한 모델의 결과를 저장할 경로를 지정합니다.
automl_rd = RD(automl) # 학습한 모델의 결과를 저장합니다.

oof_pred = automl_rd.fit_predict(tr_data, roles = roles) # fit_predict를 통해 모델을 학습하고 예측값을 반환
'''

"\n%%time \n\nautoml = TabularAutoML(task = task, \n                       timeout = TIMEOUT,\n                       cpu_limit = N_THREADS,\n                       reader_params = {'n_jobs': N_THREADS, 'cv': N_FOLDS, 'random_state': RANDOM_STATE},\n                       general_params = {'use_algos': [['lgb', 'cb']]},\n                       lgb_params = {'default_params': lgb_params, 'freeze_defaults': True},\n                       cb_params = {'default_params': cb_params, 'freeze_defaults': True}\n                      ) # 모델 생성\n\nRD = ReportDeco(output_path = 'tabularAutoML_model_report') # 학습한 모델의 결과를 저장할 경로를 지정합니다.\nautoml_rd = RD(automl) # 학습한 모델의 결과를 저장합니다.\n\noof_pred = automl_rd.fit_predict(tr_data, roles = roles) # fit_predict를 통해 모델을 학습하고 예측값을 반환\n"

**`TabularAutoML` 설정(모델 설계도)**
: 내가 정한 파라미터로 lgb랑 cb를 써서 자동으로 학습시키기 위함.  
- `general_params = {'use_algos': [['lgb', 'cb']]}`: 두 모델 같이 쓰겠단 뜻
- `freeze_defaults: True`: 직접 찾은 최적 파라미터 강제로 사용하겠단 뜻  

**`ReportDeco`: 리포트 생성기**  
학습이 끝나고 나서 "어떤 변수가 중요했는지", "성능은 어땠는지"를 HTML 형태의 보고서로 예쁘게 저장해주는 기능.  

**`fit_predict`: 학습과 검증 동시에**  
보통은 fit만 하는데 ...
OOF(Out-Of-Fold) 예측으로 여러조각으로 나눈 전체 데이터를 일부는 학습+남은부분은 예측하는 과정을 반복.  
그렇게 반환한 값은 학습 데이터는 전체에 대한 '모델의 진솔한 예측값'이 됨.  